# 09 — Optional English-trained anchor for forgetting

**Purpose:** Create one shared English-specialized model from which BOTH LoRA and full Urdu adaptation begin.

**Platform:** Kaggle or Colab (Python + GPU for model stages). Never use held-out test data for model selection.

In [ ]:
from pathlib import Path
import os, sys
CANDIDATES = [Path.cwd(), Path.cwd().parent, Path('/kaggle/working/Urdu_Qwen_Research_Notebooks'), Path('/content/Urdu_Qwen_Research_Notebooks')]
ROOT = next((p for p in CANDIDATES if (p / 'project_utils.py').exists()), None)
if ROOT is None:
    raise FileNotFoundError('Unzip the entire project folder into /kaggle/working/ or /content/, then run this notebook. See README.md.')
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
from project_utils import *
folders()
print('Project folder:', ROOT)

A direct before/after English score from the original Qwen model is only a language-retention check, not necessarily meaningful *catastrophic forgetting*. For a stronger claim, establish English ability first using the SIB-200 English training split. Then save a **shared English-trained checkpoint**, and start both Urdu training methods from that checkpoint. The resulting comparison isolates what Urdu adaptation changes.

In [ ]:
import torch
RUN_ANCHOR=False  # English training consumes additional GPU time
if RUN_ANCHOR:
    assert torch.cuda.is_available()
    r=train_run('lora','sib200',size=10,seed=42,lang='en',rank=8,
                run_name='sib200_english_anchor_lora_r8')
    print('English adapter ready:',r['saved_path'])

## Merge the English LoRA adapter for a common starting point

This modifies the base weights once, creating a new English-adapted checkpoint. Then both Urdu methods begin from the same saved merged model. **Do not mix the English adapter and Urdu adapter in a way that gives only one method a hidden advantage.**

In [ ]:
MERGE_ANCHOR=False
if MERGE_ANCHOR:
    from peft import PeftModel
    r=read_json(ROOT/'outputs/checkpoints/sib200_english_anchor_lora_r8/complete.json')
    m,t=load_model(r['base_model'],adapter_path=r['saved_path'])
    merged=m.merge_and_unload()
    target=ROOT/'outputs/english_anchor_merged'
    target.mkdir(exist_ok=True)
    merged.save_pretrained(str(target),safe_serialization=True)
    t.save_pretrained(str(target))
    print('Saved common anchor:',target)
    print('Set MODEL_OVERRIDE=str(ROOT/"outputs/english_anchor_merged") in notebooks 03 and 04.')

### Cross-lingual evaluation

Use notebook 05 with `LANG='en'` and `SPLIT='test'` to compare the English anchor **before** Urdu adaptation and each Urdu-trained model **after** adaptation. Your final report should distinguish English retention from broader catastrophic forgetting.